# TV2 — Data Profiling 12 file Parquet
Notebook hướng dẫn sử dụng script `profile_12_parquet.py` và đọc kết quả. Trước khi chạy, đặt ba ZIP vào `data/zips/` theo README.

**Chú ý:** Chạy trên máy có thể cài `duckdb`. Không suy diễn số bản ghi thành số nhân viên.

## Bước 1 — Cài đặt
Chạy cell dưới để cài thư viện. Nếu đã cài, có thể bỏ qua.

In [ ]:
%pip install duckdb pandas matplotlib

## Bước 2 — Chạy profiling trên dữ liệu thật
Script tự xử lý cả 12 file, mỗi file một lượt. Nếu notebook báo không tìm thấy script, hãy mở notebook từ đúng thư mục.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "profile_12_parquet.py"], check=True)

## Bước 3 — Đọc bảng tổng quan
Kiểm chứng 12 file và số bản ghi, cột.

In [ ]:
from pathlib import Path
import pandas as pd
out = Path("reports/profiling")
summary = pd.read_csv(out / "file_summary.csv")
display(summary)
print("Số file:", len(summary), "| Tổng bản ghi:", summary["rows"].sum())

## Bước 4 — Cột NULL / REDACTED nhiều nhất
Đọc `column_profile.csv`. Giá trị DISTINCT là **ước lượng**, không phải kết quả đếm chính xác.

In [ ]:
cols = pd.read_csv(out / "column_profile.csv")
display(cols.sort_values("null_pct", ascending=False).head(15))
display(cols.sort_values("redacted_pct", ascending=False).head(15))

## Bước 5 — Kiểm tra count và thời gian
`count` là chuỗi ở nguồn. Đừng biến REDACTED thành 0. Khác tháng hiệu lực với tháng file không mặc nhiên là lỗi.

In [ ]:
count = pd.read_csv(out / "count_quality.csv")
month = pd.read_csv(out / "month_consistency.csv")
display(count)
display(month)

## Bước 6 — Phân bố dữ liệu
Bảng dưới xếp hạng theo **số dòng** (record_count), không nhất thiết là số nhân sự hay số lượt sự kiện.

In [ ]:
cat = pd.read_csv(out / "category_top10.csv")
display(cat[(cat.dataset=="employment") & (cat.period==202506) & (cat.column=="agency_code")])

## Bước 7 — Chốt phát hiện để bàn giao TV3
Viết ngắn gọn từng vấn đề theo cấu trúc: **Field → Evidence → Risk → Proposed ETL Rule → Status (Verified/Need codebook)**. Dựa trên CSV thực, không tự đặt số liệu.

Bước sau là EDA chuyên sâu (phân bố 4 kỳ, Pareto, cross-tab, mối liên hệ).